# Synthetic Data Generation for Stochastic Attention Experiments
In this notebook, we generate unit-sphere pattern datasets for Experiments 1--3. Each dataset is a matrix whose columns are i.i.d. uniform on the unit sphere.

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
>
> * __Generate synthetic unit-sphere data:__ Produce random memory matrices whose columns lie on the unit sphere with controlled dimension d and pattern count K
> * __Parameterize experiments by load ratio:__ Understand how the ratio K/d governs the difficulty of pattern retrieval and set up sweeps across load ratios
> * __Ensure reproducibility:__ Use explicit random seeds and on-disk storage (JLD2) so that all downstream experiments draw from identical datasets

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

___
## Experiment 1: Temperature Spectrum
Generate 5 independent datasets with $d = 64$ and $K = 16$ stored patterns.

In [2]:
result_exp1 = datagenerate(64, 16, 5;
    path = joinpath(_PATH_TO_DATA, "exp1_temperature"),
    seed = 11111
);
println("Experiment 1: Generated $(result_exp1["N"]) datasets of size $(result_exp1["d"]) × $(result_exp1["K"])")
println("Column norms (dataset 1): ", [round(norm(result_exp1["datasets"][1][:, k]); digits=6) for k in 1:result_exp1["K"]])

Experiment 1: Generated 5 datasets of size 64 × 16
Column norms (dataset 1): [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


___
## Experiment 2: Convergence Diagnostics
Generate 5 independent datasets with $d = 8$ and $K = 4$ (small $d$ so the ground-truth Boltzmann density is tractable).

In [3]:
result_exp2 = datagenerate(8, 4, 5;
    path = joinpath(_PATH_TO_DATA, "exp2_convergence"),
    seed = 22222
);
println("Experiment 2: Generated $(result_exp2["N"]) datasets of size $(result_exp2["d"]) × $(result_exp2["K"])")
println("Column norms (dataset 1): ", [round(norm(result_exp2["datasets"][1][:, k]); digits=6) for k in 1:result_exp2["K"]])

Experiment 2: Generated 5 datasets of size 8 × 4
Column norms (dataset 1): [1.0, 1.0, 1.0, 1.0]


___
## Experiment 3: Load Ratio Sweep
Fix $d = 64$ and generate datasets at load ratios $K/d \in \{0.1, 0.25, 0.5, 1, 2, 4, 8\}$.
Each ratio gets 5 independent datasets.

In [4]:
d_exp3 = 64
load_ratios = [0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0]
results_exp3 = Dict{Float64, Dict{String, Any}}()

for (i, ratio) in enumerate(load_ratios)
    K_i = max(1, round(Int, ratio * d_exp3))  # K = ratio × d, at least 1
    seed_i = 33000 + i
    result = datagenerate(d_exp3, K_i, 5;
        path = joinpath(_PATH_TO_DATA, "exp3_loadratio", "ratio_$(ratio)"),
        seed = seed_i
    )
    results_exp3[ratio] = result
    println("  K/d = $(ratio) → K = $(K_i), generated $(result["N"]) datasets")
end
println("\nExperiment 3: Done.")

  K/d = 0.1 → K = 6, generated 5 datasets
  K/d = 0.25 → K = 16, generated 5 datasets
  K/d = 0.5 → K = 32, generated 5 datasets
  K/d = 1.0 → K = 64, generated 5 datasets
  K/d = 2.0 → K = 128, generated 5 datasets
  K/d = 4.0 → K = 256, generated 5 datasets
  K/d = 8.0 → K = 512, generated 5 datasets

Experiment 3: Done.


___
## Experiment 3 (continued): Fill in Additional Load Ratios
Add intermediate ratios for a denser sweep. Seeds start at 33100 to avoid collision with the original 33001–33007.


In [ ]:
# Additional load ratios to fill in the sweep
extra_ratios = [0.05, 0.15, 0.35, 0.75, 1.5, 3.0, 6.0, 10.0]

for (i, ratio) in enumerate(extra_ratios)
    K_i = max(1, round(Int, ratio * d_exp3))
    seed_i = 33100 + i  # non-overlapping with original 33001–33007
    ratio_dir = joinpath(_PATH_TO_DATA, "exp3_loadratio", "ratio_$(ratio)")
    
    if isdir(ratio_dir)
        println("  K/d = $ratio already exists, skipping")
        continue
    end
    
    result = datagenerate(d_exp3, K_i, 5;
        path = ratio_dir,
        seed = seed_i
    )
    println("  K/d = $ratio → K = $K_i, generated $(result["N"]) datasets (seed=$seed_i)")
end
println("\nAdditional ratios: Done.")


___
## Summary
We generated all synthetic datasets needed for the stochastic attention experiments.

> __Key Takeaways:__
>
> * **Unit-sphere patterns:** All memory matrices have columns drawn uniformly from the unit sphere, ensuring consistent geometry across experiments
> * **Three experiment configurations:** Temperature spectrum (d=64, K=16), convergence diagnostics (d=8, K=4), and load-ratio sweep (d=64, K/d from 0.05 to 10)
> * **Reproducibility:** Each dataset is generated with a fixed random seed and saved to disk in JLD2 format for exact reproducibility across notebook runs

___

In [5]:
# Walk the data directory and list everything we just created
for (root, dirs, files) in walkdir(_PATH_TO_DATA)
    for f in files
        println(relpath(joinpath(root, f), _PATH_TO_DATA))
    end
end

exp1_temperature/data.jld2
exp1_temperature/patterns_1.csv
exp1_temperature/patterns_2.csv
exp1_temperature/patterns_3.csv
exp1_temperature/patterns_4.csv
exp1_temperature/patterns_5.csv
exp2_convergence/data.jld2
exp2_convergence/patterns_1.csv
exp2_convergence/patterns_2.csv
exp2_convergence/patterns_3.csv
exp2_convergence/patterns_4.csv
exp2_convergence/patterns_5.csv
exp3_loadratio/ratio_0.1/data.jld2
exp3_loadratio/ratio_0.1/patterns_1.csv
exp3_loadratio/ratio_0.1/patterns_2.csv
exp3_loadratio/ratio_0.1/patterns_3.csv
exp3_loadratio/ratio_0.1/patterns_4.csv
exp3_loadratio/ratio_0.1/patterns_5.csv
exp3_loadratio/ratio_0.25/data.jld2
exp3_loadratio/ratio_0.25/patterns_1.csv
exp3_loadratio/ratio_0.25/patterns_2.csv
exp3_loadratio/ratio_0.25/patterns_3.csv
exp3_loadratio/ratio_0.25/patterns_4.csv
exp3_loadratio/ratio_0.25/patterns_5.csv
exp3_loadratio/ratio_0.5/data.jld2
exp3_loadratio/ratio_0.5/patterns_1.csv
exp3_loadratio/ratio_0.5/patterns_2.csv
exp3_loadratio/ratio_0.5/patterns_3